In [1]:
import sys
import types
import torch
import torch.nn as nn

sys.path.insert(0, ".")
from DialogueGraph import (
    _SelfAttentivePooling,
    _UtteranceFeatureExtractor,
    _DialogueGCNLayer,
    _DialogueGCN,
    _ContextAttention,
    DialogueGraph,
)

# Lightweight BERT stub: returns last_hidden_state of shape (B, seq_len, 768)
class _FakeBERT(nn.Module):
    def forward(self, input_ids, attention_mask=None, **kw):
        B, S = input_ids.shape
        out  = types.SimpleNamespace(last_hidden_state=torch.randn(B, S, 768))
        return out

# Lightweight WavLM stub: returns 13 hidden states of shape (B, T_wav, 768)
# WavLM subsamples ~50x; keep T_wav small for speed
_T_WAV = 32
class _FakeWavLM(nn.Module):
    def forward(self, input_values, attention_mask=None, output_hidden_states=False, **kw):
        B = input_values.shape[0]
        hidden_states = tuple(torch.randn(B, _T_WAV, 768) for _ in range(13))
        return types.SimpleNamespace(hidden_states=hidden_states)

# Tokenizer stub: returns fixed-length token tensors on CPU
class _FakeTokenizer:
    def __call__(self, texts, return_tensors, padding, truncation, max_length):
        B   = len(texts)
        seq = 12  # fixed short sequence for tests
        return {
            "input_ids":      torch.zeros(B, seq, dtype=torch.long),
            "attention_mask": torch.ones(B, seq, dtype=torch.long),
        }

# Processor stub: returns fixed input_values on CPU
class _FakeProcessor:
    def __call__(self, audio_list, sampling_rate, return_tensors, padding):
        B        = len(audio_list)
        n_samples = 400  # arbitrary raw sample length before WavLM
        return {"input_values": torch.randn(B, n_samples)}

SAMPLE_RATE = 16_000
B, T = 2, 4          # batch size, number of turns
D_FEAT = 64          # small for fast tests; d_gcn = 2*D_FEAT = 128
D_OUT  = 256
N_CTX  = T - 1       # context turns (all but anchor)

fake_bert     = _FakeBERT()
fake_wavlm    = _FakeWavLM()
fake_tokenizer = _FakeTokenizer()
fake_processor = _FakeProcessor()

print("Stubs ready.")


ImportError: cannot import name '_ContextAttention' from 'DialogueGraph' (c:\Users\jackm\Documents\Github Projects\imgs789-project_graph-dialogue-modeling\src\DialogueGraph.py)

In [ ]:
pool = _SelfAttentivePooling(dim=D_FEAT)

# basic shape: (B, T, d) -> (B, d)
x    = torch.randn(B, T, D_FEAT)
out  = pool(x)
assert out.shape == (B, D_FEAT), f"expected ({B}, {D_FEAT}), got {out.shape}"

# with mask: same output shape, fully-masked rows should be zero
mask = torch.ones(B, T)
mask[0, :] = 0          # batch item 0 is fully masked
out_masked  = pool(x, mask)
assert out_masked.shape == (B, D_FEAT)
assert torch.allclose(out_masked[0], torch.zeros(D_FEAT)), "fully-masked row should be zero"
assert not torch.allclose(out_masked[1], torch.zeros(D_FEAT)), "unmasked row should be non-zero"

print("PASS _SelfAttentivePooling")


In [ ]:
extractor = _UtteranceFeatureExtractor(
    bert_model  = fake_bert,
    wavlm_model = fake_wavlm,
    tokenizer   = fake_tokenizer,
    processor   = fake_processor,
    sample_rate = SAMPLE_RATE,
    d_feat      = D_FEAT,
)

N_SAMPLES = 8_000
audio     = torch.randn(B, T, N_SAMPLES)
lengths   = torch.full((B, T), N_SAMPLES, dtype=torch.long)
texts     = [[f"speaker {b} turn {t}" for t in range(T)] for b in range(B)]

out = extractor(audio, lengths, texts)
assert out.shape == (B, T, 2 * D_FEAT), f"expected ({B}, {T}, {2*D_FEAT}), got {out.shape}"

# with text_only flag: audio turns flagged True should produce zero audio embedding
text_only = torch.zeros(B, T, dtype=torch.bool)
text_only[:, 1] = True      # turn 1 is text-only for all batch items
out_to = extractor(audio, lengths, texts, text_only=text_only)
assert out_to.shape == (B, T, 2 * D_FEAT)

print("PASS _UtteranceFeatureExtractor")


In [ ]:
# 2-speaker conversation: A B A B (4 context turns)
spk  = [["A", "B", "A", "B"], ["A", "A", "B", "B"]]
N    = 4
adj  = _DialogueGCNLayer.build_adjacency(spk, N, device=torch.device("cpu"))

assert adj.shape == (2, _DialogueGCNLayer.NUM_RELATIONS, N, N), \
    f"expected (2, 5, {N}, {N}), got {adj.shape}"

# self-loops must be on for every node
self_rel = _DialogueGCNLayer._SELF
assert adj[0, self_rel].diag().all(), "self-loops missing in batch 0"
assert adj[1, self_rel].diag().all(), "self-loops missing in batch 1"

# off-diagonal entries must not appear in the SELF relation
off_diag_mask = ~torch.eye(N, dtype=torch.bool)
assert not adj[0, self_rel][off_diag_mask].any(), "SELF relation has off-diagonal entries"

# for batch 0 (A B A B): turns 0 and 2 are same speaker (A)
# edge 0->2 (past->future, same speaker) -> INTRA_P2F, so adj[b, INTRA_P2F, v=2, u=0] == 1
intra_p2f = _DialogueGCNLayer._INTRA_P2F
assert adj[0, intra_p2f, 2, 0] == 1.0, "intra past->future edge (0->2) missing"

# edge 2->0 (future->past, same speaker) -> INTRA_F2P, adj[b, INTRA_F2P, v=0, u=2] == 1
intra_f2p = _DialogueGCNLayer._INTRA_F2P
assert adj[0, intra_f2p, 0, 2] == 1.0, "intra future->past edge (2->0) missing"

# turns 0 and 1 are different speakers -> inter edges
inter_p2f = _DialogueGCNLayer._INTER_P2F
assert adj[0, inter_p2f, 1, 0] == 1.0, "inter past->future edge (0->1) missing"

# every cell must be binary
assert set(adj.unique().tolist()).issubset({0.0, 1.0}), "adjacency should be binary"

print("PASS _DialogueGCNLayer.build_adjacency")


In [ ]:
D_GCN = 2 * D_FEAT
layer = _DialogueGCNLayer(d_model=D_GCN)

h   = torch.randn(B, N_CTX, D_GCN)
spk = [["A", "B", "A"], ["A", "B", "B"]]   # N_CTX = 3
adj = _DialogueGCNLayer.build_adjacency(spk, N_CTX, device=torch.device("cpu"))
out = layer(h, adj)

assert out.shape == (B, N_CTX, D_GCN), f"expected ({B}, {N_CTX}, {D_GCN}), got {out.shape}"
assert not torch.isnan(out).any(), "NaN in GCN layer output"

print("PASS _DialogueGCNLayer.forward")


In [ ]:
gcn = _DialogueGCN(d_model=D_GCN, num_layers=1)

h   = torch.randn(B, N_CTX, D_GCN)
spk = [["A", "B", "A"], ["A", "B", "B"]]
out = gcn(h, spk)

assert out.shape == (B, N_CTX, D_GCN), f"expected ({B}, {N_CTX}, {D_GCN}), got {out.shape}"
assert not torch.isnan(out).any()

# stacking 2 layers should not change output shape
gcn2 = _DialogueGCN(d_model=D_GCN, num_layers=2)
out2 = gcn2(h, spk)
assert out2.shape == (B, N_CTX, D_GCN), "2-layer GCN shape mismatch"

print("PASS _DialogueGCN")
